# Fase 4 — Agências de Fact-Checking a partir da URL

Identifica a agência de origem de cada notícia falsa a partir do domínio/caminho da coluna `URL`.

In [ ]:
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

CORAL = "#EF476F"   # Fake
AZUL = "#118AB2"    # Real

mpl.rcParams.update({
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "font.family": "sans-serif",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.titlecolor": "#073B4C",
    "axes.labelsize": 10.5,
    "axes.labelweight": "bold",
    "axes.edgecolor": "#999999",
    "axes.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": "#E6E6E6",
    "grid.linewidth": 0.7,
    "legend.frameon": False,
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
})

df = pd.read_csv("../dados/fakerecogna_bruto.csv")
df = df.dropna(subset=["Noticia", "Classe", "Categoria", "URL"]).reset_index(drop=True)
df["Classe"] = df["Classe"].astype(int)

In [ ]:
def identificar_fonte(url):
    if pd.isna(url):
        return "Não identificado"
    url = str(url).lower()
    if "projetocomprova" in url or "/comprova/" in url:
        return "Projeto Comprova"
    if "/confere/" in url:
        return "UOL Confere"
    if "boatos.org" in url:
        return "Boatos.org"
    if "aosfatos" in url:
        return "Aos Fatos"
    if "e-farsas" in url:
        return "E-farsas"
    if "checamos.afp" in url or "afp-brasil" in url:
        return "AFP Checamos"
    if "fato-ou-fake" in url:
        return "Fato ou Fake (G1)"
    if "estadao" in url and "verifica" in url:
        return "Estadão Verifica"
    if "lupa" in url:
        return "Agência Lupa"
    if "g1.globo.com" in url:
        return "G1"
    if "extra.globo.com" in url:
        return "Extra"
    if "gov.br" in url:
        return "Ministério da Saúde"
    if "uol.com.br" in url:
        return "UOL"
    return "Outra fonte (verificar)"

df["fonte"] = df["URL"].apply(identificar_fonte)
df.loc[df["Classe"] == 0, "fonte"].value_counts()

**Nota metodológica:** a detecção usa o caminho da URL (não só o domínio) — `/comprova/` e `/confere/` capturam Projeto Comprova e UOL Confere mesmo quando hospedados sob domínio da UOL. Das seis agências citadas na introdução do artigo, três têm presença robusta no corpus (Boatos.org, Fato ou Fake/G1, UOL Confere); Aos Fatos, Estadão Verifica e Agência Lupa são residuais. E-farsas, AFP Checamos e Projeto Comprova completam a composição real do corpus.

## Volume por agência

In [ ]:
contagem = df.loc[df["Classe"] == 0, "fonte"].value_counts().drop(labels=["Não identificado"], errors="ignore")
contagem.index = [i.upper() for i in contagem.index]
contagem = contagem.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8.5, 5))
ax.barh(contagem.index, contagem.values, color=CORAL)
for i, v in enumerate(contagem.values):
    ax.text(v + max(contagem.values) * 0.01, i, f"{v:,}".replace(",", "."), va="center", fontsize=9, fontweight="bold")
ax.set_xlabel("NÚMERO DE NOTÍCIAS FALSAS NO CORPUS")
ax.set_title("DISTRIBUIÇÃO DAS NOTÍCIAS FALSAS POR AGÊNCIA DE ORIGEM", loc="left")
plt.tight_layout()
plt.savefig("../figuras/figura_volume_por_agencia.png")
plt.show()

## Checkpoint

In [ ]:
df.to_csv("../dados/fakerecogna_com_fonte.csv", index=False)
print("Checkpoint salvo em ../dados/fakerecogna_com_fonte.csv")